# **프로젝트**

# 프로젝트 명:
Qwen2-VL 기반 AI 생성 이미지 판별 멀티모달 파인튜닝

# 목차
A. 프로젝트 개요
  - 프로젝트명
  - 프로젝트를 들어가며
  - 사용된 개발환경, 패키지, 툴, 응용프로그램
  - 프로젝트의 목표

B. 데이터 처리
  - phase 1. 데이터 수집/저장
    - 1.1 데이터 소스 설명
    - 1.2 수집 방법
  - phase 2. 데이터 처리 및 학습 데이터셋 생성

C. 모델링 및 평가
  - phase 3. 멀티모달 파이프라인 구축
    - 3.1 베이스 모델
    - 3.2 입력포맷 설계
    - 3.3 학습 전략
  - phase 4. 멀티모달 모델 미세조정
    - 4.1 학습 데이터 구성
    - 4.2 학습 설정
    - 4.3 평가 방법
    - 4.4 결과 분석
    - 4.5 한계 및 개선 방향

D. 응용 및 배포
  - phase 5. 응용 어플리케이션
    - 5.1 서비스 개요
    - 5.2 시스템 아키텍처
    - 5.3 구현
    - 5.4 운영 고려사항

E. 마무리

# A. 프로젝트 개요

## 프로젝트명
Qwen2-VL 기반 AI 생성 이미지 판별 멀티모달 파인튜닝

## 프로젝트를 들어가며

AI 생성 이미지가 빠르게 대중화되면서 온라인에서 접하는 사진의 진짜 혹은 가짜의 경계가 점점 흐려지고 있다. SNS 콘텐츠, 쇼핑 이미지, 커뮤니티 게시물 등에서 생성형 모델로 만들어진 이미지가 자연스럽게 섞여 유통되면서 사용자는 시각적 단서만으로 해당 이미지가 AI가 생성한 것인지, 사람이 직접 촬영한 실제 사진인지 판단하기 어려운 상황을 자주 접하게 된다. 특히 생성 이미지의 품질이 높아질수록 육안으로 판별은 더 어려워지고 결과적으로 잘못된 정보 소비로 이어질 가능성도 커진다.

이미지 진위 판별 문제는 단순히 '이미지 분류'로 끝나지 않고 모델이 이미지 속 패턴을 어떻게 해석하고 판단하는지까지 포함하는 멀티모달 이해 문제로 확장되고 있다. 따라서 이미지 정보를 직접 입력으로 받아 의미 있는 판단을 내릴 수 있는 멀티모달 모델을 활용하여 AI 생성 이미지와 실사 이미지의 차이를 학습시키는 접근이 필요하다.

본 프로젝트는 사용자가 입력한 이미지가 AI 생성인지 실제로 촬영된 사진인지 판별하는 멀티모달 모델을 구축하는 것을 목표로 한다. 이를 위해 Qwen2-VL 7B Instruct를 기반 모델로 선택하고 데이터셋을 학습에 적합한 형태로 전처리한 후 멀티모달 파인튜닝을 수행하여 이미지 진위 판별 성능을 향상시키고자 한다.

### 사용된 개발환경, 패키지, 툴, 응용프로그램

프로그래밍 언어: Python3 <br>
데이터 전처리: pandas <br>
이미지 처리: PIL <br>
기본 멀티모달 모델(파인튜닝 베이스): Qwen/Qwen2-VL-7B-Instruct (Vision-to-Seq) <br>
멀티모달 입출력 처리(Processor): transformers, qwen_vl_utils <br>
모델 학습/미세조정(LoRA/SFT): torch, transformers, peft, trl <br>

# 프로젝트에 고려할 수 있는 AI 서비스 & API (유, 무료)

- Hugging Face(website) : 사전학습(기본) 모델(Qwen/Qwen2-VL-7B-Instruct) 로드·배포, 전처리한 파인튜닝 모델(LoRA 어댑터 포함) 업로드 및 버전 관리, 오픈소스 라이브러리
- kaggle(website) : 학습용 가공 데이터셋 업로드 및 다운로드
- RunPod(website) : 멀티모달 모델 파인튜닝의 고연산 작업을 위한 GPU 인스턴스 대여

## 프로젝트의 목표
AI 생성 이미지와 실제 촬영된 사진 이미지를 포함한 데이터셋을 기반으로 Qwen2-VL 7B Instruct에 멀티모달 파인튜닝(LoRA/SFT)을 적용해, 입력 이미지의 AI 생성 여부를 빠르고 정확하게 판별하는 모델을 구축하고 이를 재현·배포 가능한 형태로 정리한다.

# B. 데이터 처리 및 분석

[데이터 전처리 code](https://github.com/eddy-fox/qwen2vl-finetuning-ai-image-detector/blob/main/preprocessing.ipynb)

## Phase1. 데이터 수집 및 저장

### 1.1 데이터 소스 설명

- AI 생성 이미지 및 실제 촬영 이미지
    1. 출처 : Parveshiiii/AI-vs-Real(kaggle)
    2. 데이터 크기 : (13999, 2)
        - columns: ['image', 'binary_label']
        - {'image': Image(mode=None, decode=True), 'binary_label': Value('int64')}

### 1.2 데이터 수집 방법
Kaggle에서 제공되는 Parveshiiii/AI-vs-Real 공개 데이터셋을 활용

## phase 2. 데이터 처리 및 학습 데이터셋 생성

(1) 데이터셋 다운로드 후 클래스라벨(binary_label)을 기준으로 train-test split을 사용하여 학습용, 테스트용 데이터 분리 <br>
    - 실무에서 프로토타이핑시 하루 안에 처리할 수 있는 양을 기준으로 하여 ai 생성 이미지 1000장, 실제 촬영된 사진 1000장 총 2000장으로 데이터셋을 구성한 후 8:2로 train-test를 분할<br>
(2) 해당 이미지 다운로드 <br>
(3) 상대경로를 사용하여 img_path열 생성 <br>
(4) split(train/test)열을 만든 후 binary_label열을 붙여 ai_vs_real_meta.json 생성 <br>
(5) Kaggle에 학습데이터 업로드 <br>

# C. 모델링 및 평가

[모델 학습 및 평가CODE](https://github.com/eddy-fox/qwen2vl-finetuning-ai-image-detector/blob/main/training_and_evaluation.ipynb)

## phase 3. 멀티모달 파이프라인 구축

### 3.1 베이스 모델

- Qwen/Qwen2-VL-7B-Instruct

    - 이미지+텍스트 입력을 함께 이해하고 텍스트로 응답하는 VLM(Vision-to-Seq)
    - 지시문 형태의 출력 포맷을 통제하기 쉬움
    - AutoProcessor / Qwen2VLProcessor + process_vision_info 워크플로우로 멀티모달 입력 포맷 변환이 표준화되어 학습/추론 파이프라인 구현이 수월함
    - 7B급 모델 : 단일 GPU 환경에서도 LoRA/SFT로 파인튜닝 실험을 수행하기에 현실적인 크기

### 3.2 입력데이터 포맷

Qwen2-VL-7B-Instruct의 멀티모달 채팅 입력 규격에 맞추어 이미지 1장과 텍스트 지시문을 함께 제공하고 모델이 이진 라벨(AI/Real)을 출력하도록 입력 포맷을 설계했다. 데이터셋의 각 샘플은 messages 기반(OpenAI Chat Completions 스타일)의 구조로 변환해 학습·추론에서 동일한 포맷을 재사용하도록 구성했다.

- 메시지 구조(3-turn 구성)
    - System: 모델의 역할과 출력 규칙을 정의하는 시스템 메시지(system_message)를 텍스트로 삽입
    - User: 
        - 텍스트 프롬프트(prompt)와 입력 이미지(sample["file_path"])를 하나의 턴에 함께 포함
        - content를 리스트로 구성해 {"type":"text"}, {"type":"image"}를 순서대로 제공
    - Assistant: 정답 라벨을 텍스트로 제공하여 sample["binary_label"]을 타겟 출력으로 사용

- 채팅 템플릿 적용
    - processor.apply_chat_template()를 사용해 messages 구조를 Qwen2-VL이 이해하는 텍스트 시퀀스로 변환
    - 동일한 템플릿을 기준으로 텍스트 입력이 생성되도록 구성하여 학습·추론 단계에서 포맷 일관성을 유지

- 멀티모달 입력 패킹(이미지 처리 파이프라인)
    - process_vision_info(messages)로 messages에서 이미지 입력을 추출
    - 이미지 파일 경로를 PIL.Image로 로드한 뒤, 입력 크기 관리를 위해 최대 변 길이를 1024px로 제한하여 리사이즈
    - processor(text=[], images=[], padding=True, return_tensors="pt") 형태로 텍스트·이미지를 함께 텐서로 패킹해 모델 입력으로 사용

### 3.3 학습 전략

1) 지도 미세조정(SFT) 목표 정의(라벨 출력 고정)
    - Qwen2-VL-7B-Instruct가 입력 이미지에 대해 AI 또는 Real 같은 짧은 정답 라벨을 일관되게 출력하도록 학습 목표를 설정
    - 분류 헤드 추가 대신 라벨 텍스트를 생성하는 형태로 태스크를 구성해 SFT로 정답 출력 패턴을 학습 <br>

2) 미세조정 방식 선택(PEFT/LoRA 적용)
    - 7B급 멀티모달 모델을 제한된 GPU 자원에서 효율적으로 학습하기 위해 LoRA 기반 PEFT를 적용
    - 학습 비용(메모리,시간)을 줄이면서도 태스크 적응에 필요한 파라미터만 업데이트하도록 설계 <br>

## phase 4. 멀티모달 모델 미세조정

### 4.1 학습 데이터 구성

- 데이터 생성 개요
    - 학습 데이터는 Kaggle에 업로드한 이미지 데이터셋을 기반으로 AI 생성 vs 실사(Real) 이진 분류를 목표로 구성
    - 각 샘플은 이미지 1장 + 텍스트 지시문(prompt) + 정답 라벨(binary_label) 로 구성되며, 멀티모달 SFT 학습을 위해 messages(system-user-assistant) 형식으로 변환하여 사용

1.  데이터 및 메타데이터 로드
    - Kaggle에서 다운로드한 이미지와 메타데이터를 DataFrame으로 로드

2. 이미지 시각화 검토(샘플 품질 확인)
    - display_image_batch(df, start_idx, batch_size)로 배치 단위 이미지 그리드를 출력하여 데이터가 정상 로드되는지 및 라벨이 의도와 맞는지 육안으로 검토
    - 특정 클래스만 필터링하여(binary_label) 샘플을 집중 확인할 수 있도록 구성 <br>

3. 샘플 식별자(file_id) 설정
    - 학습/평가에서 특정 샘플을 재현 가능하게 추적하기 위해 file_id를 부여
    - 시각화 시 file_id, file_path, binary_label을 함께 표시하여 샘플 단위로 확인 및 참조가 가능하도록 메타데이터를 정리 <br>
    
4. 학습 라벨(binary_label) 확정
    - 원본 데이터셋에 binary_label이 이미 포함되어 있음 
        - AI 생성 이미지 = 0, 실제 촬영 이미지 = 1
    - 별도의 라벨 재매핑 없이 binary_label 값을 그대로 정답 타깃으로 사용
    - 학습 데이터 구성 시 messages의 assistant content에 binary_label을 포함하여 모델이 입력 이미지에 대해 0 또는 1을 출력하도록 지도 미세조정(SFT)을 수행 <br>

5. 학습 입력 포맷(messages) 변환
    - 각 샘플을 Qwen2-VL 입력 규격에 맞춰 messages 구조로 변환
        - system: 출력 규칙/역할 정의(system_message)
        - user: 지시문 텍스트(prompt) 및 이미지(file_path)
        - assistant: 정답 라벨(binary_label)

    - 변환된 결과를 train_dataset, test_dataset 형태로 구성하여 학습/평가에 사용 <br>

6. 학습/평가용 데이터셋 구성
    - 데이터 전처리 단계에서 각 샘플에 split(train/test) 값을 미리 부여해 두었기 때문에 학습 단계에서는 별도로 랜덤 분할을 하지 않고 Hugging Face Dataset에서 filter()로 분리

### 4.2 학습 설정

#### 4.2.1 프롬프트 구성

- System 메시지로 모델 역할을 “AI 생성 이미지 vs 실제 사진 판별”로 고정
- User 프롬프트에서 출력 형식을 0 또는 1 숫자 1개만 반환하도록 강제
- 0: AI 생성 이미지
- 1: 실제 촬영 이미지

#### 4.2.2 미세조정 방식(PEFT: LoRA)

- LoRA 설정 : 
    - lora_alpha=16,
    - lora_dropout=0.05,
    - r=8,
    - bias="none",
    - target_modules=["q_proj", "v_proj"],
    - task_type="CAUSAL_LM"

#### 4.2.3 학습 하이퍼파라미터(SFTConfig)

- 학습 프레임워크: trl의 SFTTrainer
- 주요 설정:
    - num_train_epochs=3
    - per_device_train_batch_size=8
    - gradient_accumulation_steps=8
    - Optimizer: adamw_torch_fused
    - Learning rate: 2e-4
    - LR scheduler: constant
    - Warmup: warmup_ratio=0.03
    - Gradient clipping: max_grad_norm=0.3

- 로깅/저장 : 
    - logging_steps=5
    - save_strategy="epoch"
    - 기록: report_to="wandb"

- 정밀도 및 연산 설정 : 
    - Mixed precision: bf16=True
    - TensorFloat-32: tf32=True
    - 메모리 절감: gradient_checkpointing=True
        - 추가 설정: gradient_checkpointing_kwargs={"use_reentrant": False}

#### 4.2.4 멀티모달 배치구성

- 이미지 입력은 학습시 OOM을 피하기 위해 최대 변 길이를 768px로 제한(1024 사용시 OOM 발생)
- processor.apply_chat_template()로 텍스트를 생성하고 process_vision_info()로 이미지 입력을 추출하여 processor(text, images, padding=True)로 배치 텐서를 구성
- 손실 계산에서 제외할 토큰 처리
    - 패딩 토큰(pad_token_id)은 -100으로 마스킹
    - 이미지 토큰(image_tokens)도 -100으로 마스킹하여 텍스트(라벨) 예측에만 손실이 반영되도록 설정

#### 4.2.5 학습 실행 및 저장
- SFTTrainer 구성:
    - train_dataset=train_dataset
    - data_collator=collate_fn
    - peft_config=peft_config(LoRA)
    - tokenizer=processor.tokenizer

### 4.3 평가 방법

본 프로젝트의 평가는 파인튜닝 전/후 성능 비교를 목표로, 동일한 테스트 데이터셋(test_dataset)에 대해 예측 라벨과 정답 라벨을 비교하여 Accuracy(정확도)를 산출하는 방식으로 진행했다.

1) 평가 대상 모델 구성(LoRA 어댑터 로드)
    - 파인튜닝 결과물은 체크포인트 형태로 저장되며 평가 시에는 특정 체크포인트의 LoRA 어댑터를 base model에 로드해 적용

2) 테스트 데이터 및 정답 라벨 추출

    - 테스트 샘플은 학습과 동일한 messages(system-user-assistant) 구조를 사용
    - 정답 라벨은 assistant 턴에 포함된 값(messages[2]["content"][0]["text"])을 기준으로 설정
        - 라벨 정의: 0 = AI 생성 이미지, 1 = 실제 촬영 이미지

3) 예측 라벨 생성
- 각 테스트 샘플에 대해 model.generate() 기반 추론 함수(generate_ai_or_real)로 예측값을 생성
- 공백 출력이 발생할 수 있어 retry_process(max_tries=10)로 재시도하여 예측 라벨 확보

4) 정확도(Accuracy) 계산

    - 각 샘플의 (정답, 예측)을 저장한 뒤, 두 값이 일치하는 경우를 집계하여 정확도를 계산
    - 동일한 테스트셋에 대해 학습 전 정확도(before_train_accuracy)와 학습 후 정확도(after_train_accuracy)를 출력해 파인튜닝 효과를 비교
    - 계산식: Accuracy(%) = (정답과 예측이 일치한 샘플 수 / 전체 테스트 샘플 수) × 100
    - 결과
        - 학습 전 정확도 : 70.25%
        - 학습 후 정확도 : 91.50%

### 4.4 결과 분석

평가 결과, 학습 전 정확도는 70.25%, 학습 후 정확도는 91.50%로 측정되어 LoRA 기반 SFT 적용 이후 모델의 이진 분류 성능이 크게 향상되었다. 즉, 데이터셋 내 분포에서는 모델이 AI가 생성한 이미지와 실제 촬영된 사진을 구분하는 규칙을 효과적으로 학습했음을 확인할 수 있다. <br>
<br>
다만 실제 이미지를 입력해 테스트했을 때, 아래 두 유형의 오분류가 반복적으로 관찰되었다.<br>

1. 실제 사진처럼 자연스러운 AI 생성 이미지를 ‘실사(1)’로 오분류
    - 프롬프트의 핵심 단서는 ‘질감/디테일의 비현실성’, ‘기하학/구조 오류’, ‘조명 불일치’, ‘워터마크’ 등이다. 그런데 최근 생성 이미지 중 일부는 위 단서들이 거의 나타나지 않거나 매우 약하게 나타나 실사와 구분 신호가 희미해진다.
    - 결과적으로 모델 입장에서는 명확한 AI 단서가 약한 이미지를 실제 촬영된 사진으로 오판할 수 있고, 이는 기준이 틀렸다기보다 기준을 적용할 단서가 부족한 케이스로 해석할 수 있다.

2. 실제 사진에 필터·보정 효과가 적용된 경우 ‘AI 생성(0)’으로 오분류
    - 실제 촬영 이미지라도 강한 필터(색감 변화, 과도한 샤픈, 뷰티 보정, 노이즈 제거, HDR 톤매핑 등)가 적용되면 질감이 비현실적으로 매끈해지거나 경계가 부자연스러워져 AI 생성 이미지의 특징과 유사한 패턴이 생긴다.
    - 그 결과 모델은 “필터로 인해 변형된 사진적 특성”을 생성 이미지의 단서로 해석하여 AI로 분류하는 사례가 확인되었다.

종합하면, 테스트셋 기준으로는 정확도가 크게 상승했지만, 실제 환경에서는 (1) 생성 이미지의 사실성 증가, (2) 실사 이미지의 후처리 다양성 때문에 경계 사례에서 오분류가 발생했다. 이는 모델이 AI 생성 고유 특징 뿐 아니라 시각적 패턴에도 영향을 받는다는 점을 보여주며, 향후 실제 사용을 고려할 경우 다양한 후처리/필터가 적용된 실사와 고품질 생성 이미지에 대한 추가 데이터 보강 및 평가가 필요하다.

### 4.5 한계 및 개선 방향


- 한계 1) 실제 사진과 유사한 AI 생성 이미지 부족으로 인한 일반화 한계
    - 테스트셋 정확도는 70.25%에서 91.50%로 향상했지만 실제 입력에서 실사처럼 자연스러운 AI 생성 이미지를 실제사진(1)으로 오분류하는 사례가 발생했다.
    - 이는 학습 데이터에서 실제 사진과 구분하기 생성 이미지의 비중이 낮아 프롬프트 단서(질감/기하학/조명 불일치)가 약한 케이스를 충분히 학습하지 못했기 때문인 것으로 추측한다.
- 개선 방향
    - 사진 수준의 AI 이미지 비중을 늘려 데이터 분포를 보강(다양한 생성 모델/스타일 포함)
    - 오분류된 샘플을 수집해 hard case 중심의 추가 학습 적용<br><br><br>

- 한계 2) 실제 사진의 후처리(필터/보정)로 인한 라벨 혼동
    - 실제 촬영 이미지라도 필터·보정(노이즈 제거, 샤픈, 색감 필터, 뷰티 보정 등)이 강하게 적용되면 질감이 매끈해지거나 경계가 부자연스러워져 AI 생성 이미지 특성과 유사한 패턴이 나타날 수 있다.
    - 그 결과 사람이 촬영한 사진을 AI 생성(0)으로 오분류하는 사례가 확인되었으며, 실제 서비스 환경에서는 사용자가 업로드한 사진의 후처리 편차가 커서 오류 가능성이 증가한다.

- 개선 방향
    - 필터, 보정, 압축, 리사이즈 등 후처리 변형을 포함한 실제 사진 데이터를 추가 수집하거나 증강하여 분포를 확장
    - 평가셋에도 “후처리된 실제 사진”를 별도 카테고리로 포함해 현실 환경 성능을 분리 측정<br><br><br>

- 한계 3) 워터마크/로고 단서 활용의 불안정
    - 프롬프트에는 AI로 생성된 사진을 판별시 워터마크나 로고 단서를 포함했지만 실제 테스트에서 이미지 하단에 Gemini 로고(별표시)가 존재함에도 실사(1)로 판단하는 사례가 있었다.
    - 로고가 작거나 흐리게 블렌딩되어 시각적 주목도가 낮거나 학습 데이터에 워터마크/로고가 있는 샘플이 충분히 포함되지 않아 해당 단서를 안정적으로 학습하지 못했을 가능성이 있다.

- 개선 방향
    - 워터마크/로고가 포함된 AI 생성 이미지 데이터를 의도적으로 수집해 학습·검증에 포함
    - 로고/텍스트가 포함된 샘플을 별도 태그로 관리해, 워터마크 존재 여부에 대한 오분류 케이스를 집중 분석 및 보강<br><br><br>

- 총평
    - 본 프로젝트는 Kaggle의 AI-vs-Real 이미지 데이터셋을 기반으로 Qwen2-VL-7B-Instruct를 LoRA 기반 SFT로 미세조정하여 테스트셋 정확도를 70.25%에서 91.50%로 개선하며 멀티모달 파인튜닝의 효과를 확인했다. 다만 실제 입력에서는 (1) 사진처럼 사실적인 AI 생성 이미지에서 AI 특유 단서가 약해 실제 촬영 사진으로 오판하거나, (2) 필터·보정이 적용된 실제 사진에서 질감/선명도 패턴이 왜곡되어 AI 생성으로 오판하는 경계 사례가 관찰되었다. 또한 Gemini 로고(별표시) 등 시각적 워터마크가 존재해도 이를 일관되게 단서로 활용하지 못한 사례가 있어 워터마크/후처리/고품질 생성물과 같은 현실 분포를 더 반영한 데이터 구성과 평가가 필요함을 확인했다. 향후에는 고품질 생성 이미지 및 워터마크 포함 샘플을 보강하고 후처리 변형을 포함한 실제 사진을 확장하여 경계 사례에서의 오류를 줄이는 방향으로 개선할 수 있다.

# D. 응용 및 배포

[응용프로그램CODE](https://github.com/eddy-fox/qwen2vl-finetuning-ai-image-detector/blob/main/service_example.ipynb)

## phase 5. 응용 어플리케이션

### 5.1 서비스 개요

본 프로젝트는 사용자가 이미지를 업로드하면 해당 이미지가 AI 생성 이미지(0)인지 실제 촬영 사진(1)인지 판별해주는 데모 서비스이다. UI는 Gradio로 구성했으며 사용자는 별도의 설정 없이 웹 화면에서 이미지를 올리고 즉시 판별 결과(라벨/해석)를 확인할 수 있다. 본 서비스는 파인튜닝 모델의 활용성을 검증하기 위한 간단한 단일 기능(업로드→판별) 형태로 구현했다.
- 입력: 이미지 1장(사용자 업로드)
- 출력: 예측 라벨(0/1) + 해석 문장(“AI 생성”/“실제 촬영”)

### 5.2 시스템 아키텍처

서비스는 크게 클라이언트(UI)와 추론 백엔드(모델 로딩 및 추론 함수)로 구성된다.

- Client (Gradio Web UI)
    - 사용자가 이미지를 업로드하고 결과를 확인하는 인터페이스
    - 입력 컴포넌트: gr.Image(type="pil")
    - 출력 컴포넌트: 라벨(Textbox) + 해석(Textbox)

- Inference Layer (Python Runtime)
    - Hugging Face Hub에 업로드한 모델(repo_id)에서 Processor/Model 로드
    - 업로드된 이미지를 전처리(리사이즈)한 뒤, 멀티모달 메시지(prompt+image)를 구성하여 model.generate()로 예측 수행
    - 예측 결과를 0/1로 파싱하여 사용자에게 반환

- Model Storage (Hugging Face Hub)
    - 파인튜닝된 모델 아티팩트가 저장된 원격 저장소
    - 서비스 실행 시 Hub로부터 모델을 로드하여 배포 환경에서 재현 가능하게 구성

### 5.3 구현

#### 5.3.1 모델 로딩

- Hugging Face Hub의 repo_id에서 AutoProcessor, AutoModelForVision2Seq를 로드한다.
- GPU 사용 가능 시 torch_dtype=bfloat16, device_map="auto"로 로드하여 추론 효율을 확보한다.
- model.eval()로 추론 모드로 전환한다.

#### 5.3.2 입력 전처리(이미지 리사이즈)

- 업로드된 입력은 PIL.Image로 받으며, 모델 입력 안정성을 위해 이미지의 최대 변을 MAX_SIDE=768로 제한해 리사이즈한다.
- OOM 발생 시 MAX_SIDE를 더 낮추는 방식으로 조정 가능하도록 상수로 분리했다.

#### 5.3.3 추론 로직

- messages를 system + user(text 및 image) 형태로 구성하고 processor.apply_chat_template()로 텍스트 입력을 생성한다.
- processor(text=[], images=[])로 멀티모달 배치를 만들고 model.generate()로 출력 토큰을 생성한다.
- 생성된 텍스트에서 정규표현식으로 0 또는 1을 추출하고, 이에 대응하는 해석 문장을 매핑한다.
- 결과는 (pred_label, meaning) 형태로 반환한다.

#### 5.3.4 Gradio UI 구성

- gr.Interface(fn=predict, inputs=gr.Image(), outputs=[Textbox, Textbox])로 단일 페이지 UI를 구성한다.
- 사용자는 업로드 후 즉시 “예측 라벨”과 “해석”을 확인할 수 있다.

### 5.4 운영 고려사항

1) 입력 이미지 품질 편차(후처리/압축/해상도)
    - 사용자가 업로드하는 이미지는 필터나 보정, 리사이즈, 재압축(JPEG), 스크린샷 등으로 품질 편차가 클 수 있음
    - 특히 강한 보정이 들어간 실제 사진을 AI로 오판하거나 사진처럼 사실적인 생성 이미지를 실제 촬영된 사진로 오판하는 경우가 발생할 수 있어 입력 품질에 따른 성능 변동을 전제로 운영해야 함

2) 워터마크/로고 포함 이미지 처리
    - 생성기 로고나 워터마크가 포함된 경우에도 모델이 이 정보를 단서로 활용하지 못할 수 있음
    - 운영 시 워터마크 포함 케이스를 별도 태그로 수집하고 오분류 사례를 빠르게 재현할 수 있도록 샘플을 관리하는 것이 필요

3) 성능 지표 및 모니터링 포인트
    - 단순 Accuracy 외에도 운영 목적에 따라 오탐 혹은 미탐 비용이 달라질 수 있음
    - 실제 운영에서는 업로드 로그 기반으로 오분류 유형을 분리하여 집계하고 모델 업데이트 전후로 지표를 지속 비교할 수 있어야 함

5) 자원 및 응답 시간
    - Qwen2-VL 7B는 추론 시 GPU 사용 여부에 따라 응답 시간이 크게 달라질 수 있음
    - VRAM 부족(OOM) 발생 가능성을 고려해 입력 이미지 리사이즈 크기를 조절할 수 있게 하고 필요시 CPU fallback 또는 동시 요청 제한(큐잉)을 적용하는 것도 고려해야 함

6) 재현성과 배포
    - 서비스는 Hugging Face Hub에서 모델을 로드하여 사용하기 때문에 운영 시에는 repo_id를 특정 체크포인트/리비전으로 고정하여 결과 재현성을 확보해야함
    - 모델 변경 시에는 변경 이력(버전, 날짜, 주요 수정점)과 함께 성능 비교 결과를 함께 관리해야 함

# E. 마무리

이번 프로젝트에서 가장 많은 시간을 투자한 파트는 데이터셋 선정이었다. 이미지를 직접 생성해 학습 데이터를 만들 경우 시간과 비용 부담이 컸고, 기존에 공개된 이미지 데이터셋은 최신 생성 기술을 충분히 반영하지 못하는 경우가 많았다. 특히 2024년 이후 생성형 이미지 기술이 급격히 발전하며 AI 이미지의 품질 자체가 빠르게 변했기 때문에 가능한 한 최근 경향을 반영한 데이터로 학습하는 것이 의미 있다고 판단해 데이터셋 탐색과 비교에 많은 시간을 들였다.

다만 최종 선택한 데이터셋에는 실제 촬영 사진처럼 자연스러운 AI 생성 이미지의 비중이 충분하지 않아 실제 환경에서 기대한 수준의 판별 성능을 완전히 보여주지는 못했다. 그럼에도 테스트셋 기준으로 정확도가 70.25%에서 91.50%로 상승한 결과를 통해 데이터 규모가 크지 않더라도(약 1,600장 수준) 과제 의도에 맞는 품질의 데이터가 충분히 확보된다면 멀티모달 파인튜닝만으로 성능을 유의미하게 개선할 수 있음을 확인했다.